# P100 — Requisitos para robots seguros: mediciones, análisis y nuevas conclusiones

## 1. Título y paper

**Paper:** *Requirements for Safe Robots: Measurements, Analysis and New Insights*  
**Autoría:** Sami Haddadin, Alin Albu-Schäffer, Gerd Hirzinger  
**Año y venue:** 2009 · The International Journal of Robotics Research, 28(11–12), 1507–1527  
**Nivel:** L2 · **Motor:** `seguridad_fisica`  
**Ficha completa:** [`P100_seguridad_fisica`](../../papers/foundational/P100_seguridad_fisica/README.md)

**Hito:** Sustituye la intuición sobre seguridad robótica por mediciones de impacto con maniquíes y criterios de lesión validados.

- [doi:10.1177/0278364909343970](https://doi.org/10.1177/0278364909343970)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: La seguridad de los robots industriales se resolvía con vallas: separación física total. Para trabajar junto a personas hacía falta saber qué daño produce realmente un impacto, y ese dato no existía — se legislaba y se diseñaba a ojo.
2. Ejecutar una implementación mínima de la propuesta: Medir. Impactos instrumentados con maniquíes y voluntarios, análisis de los criterios de lesión de la industria del automóvil aplicados a la robótica, y la conclusión incómoda: la masa importa menos de lo que se creía y la velocidad, mucho más.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P97
- Normativa ISO 10218 sobre robots industriales


## 4. Intuición

«Robot colaborativo» suena a categoría de producto. No lo es: la energía que un brazo puede transferir en un impacto depende de su masa y de su velocidad, y la velocidad entra al cuadrado. El mismo robot es seguro o peligroso según a qué velocidad se le haga trabajar.


## 5. Concepto mínimo

```text
E = ½·m·v²          fuerza ≈ E / distancia de frenado

    la masa entra LINEAL
    la velocidad entra al CUADRADO

⟹ frenar rinde mucho más que aligerar, y suele ser más barato
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('seguridad_fisica', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Cuánta energía lleva un brazo de 120 kg a 1,5 m/s?
2. ¿Qué reduce más la energía: dividir la masa por 10 o la velocidad por 2?
3. ¿Cuántas configuraciones superan el umbral de la zona evaluada?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('seguridad_fisica', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('seguridad_fisica', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

El brazo industrial lleva **135 J**; el mismo brazo a 0,25 m/s lleva **3,75 J** — un 2,8 % de la energía. Dividir la masa por 10 deja 13,5 J; dividir la velocidad por 2 deja 33,75 J. Y con los umbrales ilustrativos, **2 de 4** configuraciones los superan.


## 10. Comentario pedagógico

La consecuencia práctica es que la seguridad no se compra con el robot: se diseña en la célula. Límites de velocidad por zona, detección de contacto, geometría sin aristas y una evaluación de riesgos de la instalación concreta. La etiqueta del catálogo no evalúa tu tarea.


## 11. Error o anti-patrón deliberado

Anti-patrón: usar los umbrales de esta miniatura para decidir algo real.


In [ ]:
print('Los umbrales de aqui son ILUSTRATIVOS y el modelo de fuerza es grosero.')
print('Los valores normativos estan en ISO/TS 15066, dependen de la zona del cuerpo,')
print('del tipo de contacto (transitorio o con aprisionamiento) y de la geometria.')
print('Cualquier evaluacion real parte de la norma y de un analisis de riesgos.')

## 12. Corrección

Lo que sí es transferible del artículo:


In [ ]:
r = run_paper_lab('seguridad_fisica', seed=7)['result']
for c in r['configuraciones']:
    print(f"{c['escenario']:<36} {c['masa_kg']:>6} kg  {c['velocidad_ms']} m/s  "
          f"{c['energia_julios']:>7} J")
print()
print('masa /10  ->', r['reducir_masa_10x'], 'J')
print('vel. /2   ->', r['reducir_velocidad_a_la_mitad'], 'J')

## 13. Desafío guiado

Calcula a qué velocidad tendría que ir el brazo de 120 kg para llevar la misma energía que el cobot de 12 kg a 1,5 m/s.


In [ ]:
r = run_paper_lab('seguridad_fisica', seed=3)['result']
show(r)

## 14. Desafío autónomo

Toma una célula robotizada real —o descrita en un catálogo— y calcula la energía de impacto en cada fase de su ciclo. Identifica en qué fase habría que limitar la velocidad y por qué.


## 15. Evidencia de aprendizaje

Guarda la tabla de energías y tu explicación de por qué frenar rinde más que aligerar.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P100_seguridad_fisica/README.md) · evaluación formal: [`assessments/papers/P100_seguridad_fisica.md`](../../assessments/papers/P100_seguridad_fisica.md)


## 16. Cierre

La seguridad del contacto ya tiene números. Volvemos a cómo se obtiene el comportamiento: copiando a un experto, que parece lo más fácil y tiene una trampa.


## 17. Conexión con el siguiente hito



Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
